# Radar Dataset Plotter

This notebook helps visualize the synthetic radar pulse datasets generated in this project.
Use it to inspect the signal structure, PRI, frequency, amplitude, and AOA for a single pulse sequence.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")

def resolve_dataset_root():
    base = Path.cwd()
    candidates = [
        base / 'interleaved datasets',
        base.parent / 'interleaved datasets',
        base / '..' / 'interleaved datasets',
        base / 'datasets',
        base.parent / 'datasets',
        base / '..' / 'datasets',
    ]
    for c in candidates:
        if c.exists() and c.is_dir():
            return c
    raise FileNotFoundError("Could not locate the interleaved datasets folder.")

DATASET_ROOT = resolve_dataset_root()
DATASET_NAME = "interleaved datasets"
DATASET_PATH = DATASET_ROOT
SPLIT = "train"
SEQUENCE_ID = "seq_0000"
DATASET_ROOT, DATASET_PATH, SPLIT, SEQUENCE_ID


In [ ]:
def load_sequence(dataset_root: Path, split: str, sequence_id: str) -> pd.DataFrame:
    path = dataset_root / split / f"{sequence_id}.csv"
    return pd.read_csv(path).sort_values("pulse_index").reset_index(drop=True)

seq_df = load_sequence(DATASET_PATH, SPLIT, SEQUENCE_ID)
seq_df.head()


In [ ]:
def plot_sequence(seq_df: pd.DataFrame):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"Radar sequence {seq_df['sequence_id'].iloc[0]} | {len(seq_df)} pulses", fontsize=16)

    axes[0, 0].plot(seq_df["pulse_index"], seq_df["TOA"], "o-", linewidth=1.5)
    axes[0, 0].set_title("TOA vs pulse index")
    axes[0, 0].set_xlabel("Pulse index")
    axes[0, 0].set_ylabel("TOA (us)")

    axes[0, 1].plot(seq_df["pulse_index"], seq_df["Measured_Delta_TOA"], "o-", linewidth=1.5, color="tab:orange")
    axes[0, 1].axhline(seq_df["PRI"].median(), color="tab:red", linestyle="--", label="Median PRI")
    axes[0, 1].set_title("Measured Delta-TOA (PRI estimate)")
    axes[0, 1].set_xlabel("Pulse index")
    axes[0, 1].set_ylabel("?TOA (us)")
    axes[0, 1].legend()

    axes[1, 0].plot(seq_df["pulse_index"], seq_df["Frequency"], "o-", linewidth=1.5, color="tab:green")
    axes[1, 0].set_title("Frequency vs pulse index")
    axes[1, 0].set_xlabel("Pulse index")
    axes[1, 0].set_ylabel("Frequency (MHz)")

    ax_polar = fig.add_subplot(2, 2, 4, projection="polar")
    theta = np.deg2rad(seq_df["AOA"])
    scatter = ax_polar.scatter(theta, seq_df["Amplitude"], c=seq_df["Amplitude"], cmap="viridis", s=35)
    ax_polar.set_title("AOA / amplitude")
    ax_polar.set_theta_zero_location("E")
    ax_polar.set_theta_direction(1)
    ax_polar.set_rlabel_position(90)
    fig.colorbar(scatter, ax=ax_polar, pad=0.08, label="Amplitude (dB)")

    plt.tight_layout()
    plt.show()

plot_sequence(seq_df)


In [ ]:
def get_available_sequence_ids(dataset_root: Path, split: str, limit: int | None = None) -> list[str]:
    seq_dir = dataset_root / split
    file_ids = sorted(p.stem for p in seq_dir.glob("seq_*.csv"))
    if file_ids:
        return file_ids[:limit] if limit is not None else file_ids

    combined = seq_dir / "sequences.csv"
    if combined.exists():
        df = pd.read_csv(combined)
        ids = sorted(df["sequence_id"].dropna().unique().tolist())
        return ids[:limit] if limit is not None else ids

    raise FileNotFoundError(f"No sequence CSVs were found in {seq_dir}")


def plot_sequence_grid(dataset_root: Path, split: str, seq_ids: list[str], max_cols: int = 3):
    seq_ids = [seq_id for seq_id in seq_ids if (dataset_root / split / f"{seq_id}.csv").exists()]
    if not seq_ids:
        raise FileNotFoundError(f"No matching sequence files were found in {dataset_root / split}")

    n = len(seq_ids)
    rows = (n + max_cols - 1) // max_cols
    fig, axes = plt.subplots(rows, max_cols, figsize=(5 * max_cols, 4 * rows))
    axes = np.asarray(axes).ravel()

    for idx, seq_id in enumerate(seq_ids):
        seq_df = load_sequence(dataset_root, split, seq_id)
        axes[idx].plot(seq_df["pulse_index"], seq_df["TOA"], "o-", linewidth=1.2)
        axes[idx].set_title(seq_id)
        axes[idx].set_xlabel("Pulse index")
        axes[idx].set_ylabel("TOA (us)")

    for j in range(len(seq_ids), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

sample_ids = get_available_sequence_ids(DATASET_PATH, SPLIT, limit=5)
plot_sequence_grid(DATASET_PATH, SPLIT, sample_ids)


### Notes
- `TOA` is the observed time-of-arrival in microseconds.
- `Measured_Delta_TOA` is the pulse spacing and is useful for PRI analysis.
- `AOA` can be plotted in polar coordinates to expose directional structure.
- `Frequency`, `Amplitude`, and `PW` help separate regular pulses from noisy or multipath pulses.

To inspect a different dataset, change `DATASET_NAME` and rerun the cells above.
